# Phase 5 - QLoRA Decoder Fine-Tuning

This Colab notebook runs the full Phase 5 decoder QLoRA experiments on GPU hardware. Do not run the 7B-9B configs on the local RTX 3050 4GB machine.

The local diagnostic path is only for validating code paths. Full Mistral, Llama 3, and Gemma 2 results should exist only after real Colab runs complete.

## 0. GPU Check

Confirm that the Colab runtime has a GPU before starting full QLoRA training.

In [ ]:
!nvidia-smi

## 1. Mount Google Drive

Drive is used to persist Phase 5 outputs and checkpoints beyond the temporary Colab runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Recommended Drive layout:

```text
/content/drive/MyDrive/Authorship-Attribution/
  outputs/phase5/
  checkpoints/phase5/
  datasets/
```

## 2. Clone or Update Repository

Replace `<YOUR_REPO_URL>` with the GitHub repository URL before running this cell.

In [ ]:
%cd /content
!mkdir -p /content/Authorship-Attribution/repo
%cd /content/Authorship-Attribution/repo
!if [ ! -d Authorship-Attribution-in-Victorian-Periodicals ]; then git clone <YOUR_REPO_URL> Authorship-Attribution-in-Victorian-Periodicals; fi
%cd /content/Authorship-Attribution/repo/Authorship-Attribution-in-Victorian-Periodicals
!git pull

## 3. Install Dependencies

Restart the runtime after this cell if Colab reports incompatible preinstalled packages.

In [ ]:
!pip install -U pip
!pip install -r requirements.txt

## 4. Configure Colab Paths

The Phase 5 scripts use repo-relative `outputs/phase5` and `checkpoints/phase5` for active run artifacts. The existing dataset helpers use `configs/paths.local.yaml` for the processed PERIAD dataset location.

In [ ]:
%%bash
cat > configs/paths.local.yaml <<'YAML'
paths:
  workspace_root: /content/Authorship-Attribution
  repo_root: /content/Authorship-Attribution/repo/Authorship-Attribution-in-Victorian-Periodicals
  artifacts_root: /content/Authorship-Attribution/artifacts
  checkpoints_root: /content/Authorship-Attribution/checkpoints
  datasets_root: /content/Authorship-Attribution/datasets
  exports_root: /content/Authorship-Attribution/exports

runtime:
  default_environment: colab
  cloud_training_environment: colab
YAML

mkdir -p outputs/phase5 checkpoints/phase5 /content/Authorship-Attribution/datasets

## 5. Hugging Face Login

Use a token with access to gated models if running Llama 3 or Gemma 2.

In [ ]:
import os
from huggingface_hub import login

token = os.environ.get('HF_TOKEN')
if token:
    login(token=token)
else:
    login()

## 6. Restore Prior Phase 5 State From Drive

Run this before resuming interrupted jobs. It is safe if the Drive folders do not exist yet.

In [ ]:
%%bash
DRIVE_ROOT="/content/drive/MyDrive/Authorship-Attribution"
mkdir -p outputs checkpoints
if [ -d "$DRIVE_ROOT/outputs/phase5" ]; then
  mkdir -p outputs
  rsync -a "$DRIVE_ROOT/outputs/phase5" outputs/
fi
if [ -d "$DRIVE_ROOT/checkpoints/phase5" ]; then
  mkdir -p checkpoints
  rsync -a "$DRIVE_ROOT/checkpoints/phase5" checkpoints/
fi

## 7. Prepare PERIAD Dataset

`scripts/prepare_hf_datasets.py` exists in this repository, so this notebook uses that Phase 1 preparation command. The Phase 5 scripts expect processed PERIAD files and the canonical six-author `label_map.json`.

In [ ]:
!test -f scripts/prepare_hf_datasets.py && python scripts/prepare_hf_datasets.py

## 8. Mistral 7B Instruct QLoRA

This is a full Phase 5 run intended for Colab GPU hardware.

In [ ]:
!python src/training/train_decoder_qlora.py --config configs/phase5/mistral_7b_instruct_qlora.yaml
!python src/evaluation/evaluate_decoder_qlora.py --config configs/phase5/mistral_7b_instruct_qlora.yaml

## 9. Llama 3 8B Instruct QLoRA

Confirm Hugging Face gated access before running this cell.

In [ ]:
!python src/training/train_decoder_qlora.py --config configs/phase5/llama3_8b_instruct_qlora.yaml
!python src/evaluation/evaluate_decoder_qlora.py --config configs/phase5/llama3_8b_instruct_qlora.yaml

## 10. Gemma 2 9B IT QLoRA

Confirm Hugging Face gated access before running this cell.

In [ ]:
!python src/training/train_decoder_qlora.py --config configs/phase5/gemma2_9b_it_qlora.yaml
!python src/evaluation/evaluate_decoder_qlora.py --config configs/phase5/gemma2_9b_it_qlora.yaml

## 11. Resume From Checkpoint Example

Use the latest trainer checkpoint for the interrupted run. Adjust the checkpoint number and config as needed.

In [ ]:
!python src/training/train_decoder_qlora.py \
  --config configs/phase5/mistral_7b_instruct_qlora.yaml \
  --resume_from_checkpoint checkpoints/phase5/mistral_qlora/checkpoint-250

## 12. Plot and Validate Full Phase 5 Outputs

Run this only after the real Mistral, Llama 3, and Gemma 2 full runs have produced outputs. The checker uses `--require_full_runs` here so incomplete full experiments fail loudly.

In [ ]:
!python src/visualization/plot_phase5_results.py --phase5_dir outputs/phase5
!python scripts/check_phase5_outputs.py --phase5_dir outputs/phase5 --checkpoint_dir checkpoints/phase5 --require_full_runs

## 13. Persist Phase 5 Outputs and Checkpoints to Drive

Run this after each model finishes and again after final validation.

In [ ]:
%%bash
DRIVE_ROOT="/content/drive/MyDrive/Authorship-Attribution"
mkdir -p "$DRIVE_ROOT/outputs" "$DRIVE_ROOT/checkpoints"
rsync -a outputs/phase5 "$DRIVE_ROOT/outputs/"
rsync -a checkpoints/phase5 "$DRIVE_ROOT/checkpoints/"

## 14. Optional Archive for Download

Use this only for transferring completed Phase 5 artifacts. Do not commit the archive.

In [ ]:
%%bash
cd /content/Authorship-Attribution/repo/Authorship-Attribution-in-Victorian-Periodicals
zip -r /content/phase5_outputs.zip outputs/phase5
zip -r /content/phase5_checkpoints.zip checkpoints/phase5

## 15. OOM Troubleshooting

- Reduce `prompt.max_length` from `1024` to `768` or `512`.
- Increase `training.gradient_accumulation_steps` instead of batch size.
- Keep `per_device_train_batch_size: 1`.
- Use `bf16` only on GPUs that support it; otherwise switch to `fp16`.
- Reduce LoRA rank from `16` to `8` if memory remains tight.
- Resume from the latest checkpoint after runtime interruption.
- Persist `outputs/phase5` and `checkpoints/phase5` to Drive after each completed model.